# Programs and optimizers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/07_program.ipynb)

Every finite-dimensional optimization problem in minilink—controller tuning, co-design, trajectory optimization, and MPC—lowers to one canonical description, `MathematicalProgram`, solved by `Optimizer` into an `OptimizationResult`.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. `MathematicalProgram`, `Optimizer`, and `OptimizationResult`

A `MathematicalProgram` describes the nonlinear program

$$\min_{z \in \mathbb{R}^{n_z}} J(z; p) \quad \text{s.t.} \quad h(z; p) = 0,\quad g(z; p) \ge 0,\quad \text{lower} \le z \le \text{upper}.$$

It is a frozen record of pure functions of the decision vector `z`; it holds no solver state. The solver boundary is the `Optimizer`, which compiles the program once and solves it.

### `MathematicalProgram`

`MathematicalProgram(n_z, J, h=None, g=None, lower=None, upper=None, grad_J=None, hess_J=None, jac_h=None, jac_g=None, backend="numpy", metadata={}, params=None, z0=None)`

| Field | Type | Meaning |
| --- | --- | --- |
| `n_z` | `int` | Dimension of the decision vector |
| `J` | callable `J(z) -> scalar` | The objective |
| `h`, `g` | callable `-> array (n_h,)`, `-> array (n_g,)`, or `None` | Equality residual $h(z) = 0$ and inequality margin $g(z) \ge 0$ |
| `lower`, `upper` | `array (n_z,)` or `None` | Box bounds on `z`; `None` is unbounded on that side |
| `grad_J`, `hess_J` | callable or `None` | The gradient `(n_z,)` and the Hessian `(n_z, n_z)` of `J`, when written by hand |
| `jac_h`, `jac_g` | callable or `None` | The constraint Jacobians `(n_h, n_z)` and `(n_g, n_z)`, when written by hand |
| `backend` | `"numpy"` or `"jax"` | The array backend the functions are written for. On `"jax"` the missing derivatives are generated by automatic differentiation; on `"numpy"` only the derivatives given here exist, and a native method (`"gd"`, `"adam"`) requires `grad_J` |
| `metadata` | `dict` | Transcription or diagnostic notes |
| `params` | `dict` or `None` | The default values of the program's parameters. With `params` set, every function takes `(z, params)`; without, `z` alone |
| `z0` | `array (n_z,)` or `None` | The program's own start, the `Optimizer`'s default (an exported program sets it) |

The functions are native to their backend: with JAX arrays in, they return JAX expressions, with no `float(...)` cast inside.

### `Optimizer`

| Call | Returns | Meaning |
| --- | --- | --- |
| `Optimizer(program, z0=None, *, method="auto", backend=None, use_hessian=False, options=None, n_iterations=None, **method_numbers)` | `Optimizer` | Compile `program` once and bind it to a method. `z0` defaults to `program.z0`, `backend` to `program.backend`; `options` are an external solver's own knobs (`maxiter`, `ftol`); `n_iterations` fixes the step count of a native method; `**method_numbers` are a native method's numbers (`learning_rate`, `max_iterations`, `tol`, and Adam's `b1`, `b2`, `eps`, `max_grad_norm`) |
| `opt.program`, `opt.z0`, `opt.method`, `opt.backend`, `opt.n_iterations` | | What the optimizer was built with, `"auto"` resolved |
| `opt.params` | `dict` | The solve's tree, a copy: the program's parameters, and the method's numbers under `"method"` |
| `opt.program_evaluator` | `MathematicalProgramEvaluator` | The compiled program (section 3) |
| `opt.solve(*, params=None, z0=None, callback=None, verbose=False)` | `OptimizationResult` | Solve from `z0` (default `opt.z0`) at `params` (default the program's), without a recompile; `callback(z, J, t)` is called at each iterate, `t` the elapsed seconds |
| `opt.as_step_system(params=None)` | `IterationSystem`, a `StepSystem` | A native method (`"gd"`, `"adam"`) as a step system: the state is the iterate, one tick is one iteration |

The methods:

| `method` | Algorithm | Programs it admits |
| --- | --- | --- |
| `"auto"` | `"l-bfgs"` for an unconstrained or box-bounded program, `"slsqp"` for a constrained one | any |
| `"l-bfgs"` | Quasi-Newton L-BFGS-B (SciPy) | unconstrained or box-bounded |
| `"gd"`, `"adam"` | Gradient descent and Adam, projected on the box (native) | unconstrained or box-bounded |
| `"slsqp"`, `"trust-constr"` | Sequential quadratic programming, trust region (SciPy) | any |
| `"ipopt"` | Interior point (the optional `cyipopt`) | any |

### `OptimizationResult`

| Field | Type | Meaning |
| --- | --- | --- |
| `result.z` | `ndarray (n_z,)` (a JAX array on a traced solve) | The solution $z^\star$ |
| `result.success` | `bool` | The solver's convergence flag |
| `result.J` | `float \| None` | The objective $J(z^\star)$ |
| `result.message` | `str` | The solver's status message |
| `result.stats` | `dict` | Solver statistics (`"nit"`, ...) |
| `result.n_iterations` | `int \| None` | The iterations taken, `stats["nit"]` |
| `result.raw_result` | object or `None` | The backend's own result object |
| `result.solve_time_s` | `float \| None` | Wall-clock time of the backend solve; `None` on a traced solve |

## 2. Tuning a controller: `pe.program(wrt)`, `pe.pack`, and `pe.unpack`

`pe.program(wrt, bounds=None, key=0, n_trials=None, *, params=None)` exports the tuning program of a `ProblemEvaluator`: `z` packs the leaves of `pe.params` named by the dotted paths in `wrt`, `J(z)` is `pe.evaluate(params).objective`, and the rows `g(z) >= 0` are the feasibility margin (when `X` or `U` is bounded) and the reachability margin (when `Xf` is set). `bounds` maps a path to `(lower, upper)`.

In [ ]:
import numpy as np

from minilink import (
    PID,
    BallSet,
    InvertedPendulum,
    Optimizer,
    PlanningProblem,
    QuadraticCost,
)

plant = InvertedPendulum()
cost = QuadraticCost.from_system(plant, Q=np.diag([10.0, 1.0]), R=0.1 * np.eye(1))
problem = PlanningProblem(
    plant,
    cost=cost,
    x0=np.array([0.4, 0.0]),
    horizon=4.0,
    Xf=BallSet(np.zeros(2), 0.05),
)
pe = problem.compile(PID(Kp=20.0, Kd=4.0) % 0.01)

wrt = ["controller.Kp", "controller.Kd"]
program = pe.program(wrt)
print(program)

In [ ]:
opt = Optimizer(program)
best = opt.solve()
print(best)

In [ ]:
opt.method, best.z, best.success, best.J, best.n_iterations

In [ ]:
# Between the tree and the vector z
z_initial = pe.pack(pe.params, wrt)
tuned_params = pe.unpack(best.z, wrt)
z_initial, pe.pack(tuned_params, wrt)

In [ ]:
print(pe.evaluate(tuned_params))

In [ ]:
pe.loop.plot_trajectory(pe.trajectory(tuned_params));

## 3. A constrained program written by hand, and its compiled evaluator

The functions of a program are the things it is about: the objective, the equality residual, and the inequality margin.

In [ ]:
import jax.numpy as jnp

from minilink import MathematicalProgram


def J(z):
    J = 0.5 * (z @ z)

    return J


def h(z):
    h = jnp.array([z[0] + z[1] - 1.0])

    return h


def g(z):
    g = jnp.array([z[0] - 0.7])

    return g


nlp = MathematicalProgram(
    n_z=2,
    J=J,
    h=h,
    g=g,
    lower=[0.0, 0.0],
    upper=[1.0, 1.0],
    backend="jax",
)
nlp_opt = Optimizer(nlp, z0=np.array([0.5, 0.5]))
solution = nlp_opt.solve()
nlp_opt.method, solution.z, solution.success, solution.J

`nlp_opt.program_evaluator` is the compiled program, a `MathematicalProgramEvaluator`: what the solver backends call, as Python floats and NumPy arrays.

| Call | Returns | Meaning |
| --- | --- | --- |
| `pev.objective(z)` | `float` | $J(z)$ |
| `pev.grad_J(z)`, `pev.hess_J(z)` | `ndarray (n_z,)`, `(n_z, n_z)` | The gradient and the Hessian of `J`, when available (`pev.has_gradient`, `pev.has_hessian`) |
| `pev.equality_residual(z)`, `pev.inequality_margin(z)` | `ndarray (n_h,)`, `(n_g,)` | $h(z)$ and $g(z)$ |
| `pev.jac_h(z)`, `pev.jac_g(z)` | `ndarray (n_h, n_z)`, `(n_g, n_z)` | The constraint Jacobians |
| `pev.constraint_violations(z)` | `(max_eq, min_ineq, max_bound)` | The largest $\lvert h \rvert$, the least $g$ (`None` without inequalities), and the largest bound violation |
| `pev.bind(params=None)` | `None` | Bind the parameter values of the next calls: the program's defaults with `params` over them |
| `pev.n_z`, `pev.n_h`, `pev.n_g`, `pev.backend` | `int`, `int`, `int`, `str` | The program's sizes and the evaluator's backend |

In [ ]:
pev = nlp_opt.program_evaluator
(
    pev.objective(solution.z),
    pev.grad_J(solution.z),
    pev.equality_residual(solution.z),
    pev.inequality_margin(solution.z),
    pev.jac_h(solution.z),
    pev.constraint_violations(solution.z),
)

## 4. Parametric programs and differentiable fixed-count solves

With `params=` on the `MathematicalProgram`, its functions take `(z, params)` and `opt.solve(params=...)` solves at other values without a recompile. The tree a solve takes is complete: a copy of `opt.params`, edited.

In [ ]:
def J_target(z, params):
    z_bar = params["z_bar"]

    e = z - z_bar
    J = 0.5 * (e @ e)

    return J


target_program = MathematicalProgram(
    n_z=2,
    J=J_target,
    params={"z_bar": np.array([1.0, -0.5])},
    backend="jax",
)
lbfgs_opt = Optimizer(target_program, z0=np.zeros(2), method="l-bfgs")
moved = lbfgs_opt.params
moved["z_bar"] = np.array([3.0, 2.0])
lbfgs_opt.solve().z, lbfgs_opt.solve(params=moved).z

With a native method and `n_iterations=K`, a solve is exactly `K` steps: on a JAX program it is one `lax.scan`, pure, so it traces under `jit`, `vmap` and `grad`. `as_step_system()` is the same iteration as a `StepSystem`, whose rollout gives the iterates:

In [ ]:
descent = Optimizer(
    target_program,
    z0=np.zeros(2),
    method="gd",
    learning_rate=0.3,
    n_iterations=20,
)
descent.params

In [ ]:
iter_sys = descent.as_step_system()
iterates = iter_sys.compute_rollout(20)
type(iter_sys).__name__, iter_sys.n, iter_sys.p, iterates.x.shape

In [ ]:
iter_sys.plot_rollout(iterates);

The solution differentiated with respect to a parameter of the program, through the 20 steps:

In [ ]:
import jax


def z_star(z_bar):
    tree = descent.params
    tree["z_bar"] = z_bar
    z = descent.solve(params=tree).z

    return z


dz_dz_bar = jax.jacobian(z_star)(jnp.array([1.0, -0.5]))
dz_dz_bar

Next, [`08_planners`](08_planners.ipynb): `Planner`, `PlanningSolution`, `compare`, `CompiledTrajectoryPlanner`, and `ModelPredictiveController`.